<a href="https://colab.research.google.com/github/Of-Calls/sisicallcall-verification-finetuning/blob/main/titanet_medium_train_eval_colab_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TitaNet-Small Medium Train + Eval Colab v1

- 전체 Drive 마운트: `from google.colab import drive; drive.mount('/content/drive')`
- 데이터: `titanet_local_subset_medium.zip`
- 학습: TitaNet-Small / 5epoch / lr=5e-5 / batch_size=32
- 평가: baseline vs medium fine-tuned / 전체 80,000 trials
- 결과 저장:
  - 학습: `화자검증 데이터/titanet_finetune_runs/{RUN_ID}`
  - 평가: `화자검증 데이터/titanet_eval_runs/{EVAL_RUN_ID}`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
from datetime import datetime

DRIVE_DATA_DIR = Path(
    "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터"
)
DATA_ZIP_PATH = DRIVE_DATA_DIR / "titanet_local_subset_medium.zip"

BASELINE_MODEL_NAME = "titanet_small"
MAX_EPOCHS = 5
LEARNING_RATE = 5e-5
BATCH_SIZE = 16
NUM_WORKERS = 2
PRECISION = "16-mixed"
SEED = 42


MAX_AUDIO_DURATION = 8.0
MIN_AUDIO_DURATION = 0.5

LIMIT_TRAIN_ROWS = 0
LIMIT_VALID_ROWS = 0
LIMIT_TRAIN_BATCHES = 1.0
LIMIT_VAL_BATCHES = 1.0

RUN_EVAL_AFTER_TRAIN = True
MAX_TRIALS_PER_ENROLL_SEC = 0
TARGET_FARS = [0.01, 0.05, 0.10]

RUN_ID = datetime.now().strftime("titanet_medium_5epoch_lr5e5_%Y%m%d_%H%M%S")
EVAL_RUN_ID = datetime.now().strftime("eval_medium_5epoch_lr5e5_%Y%m%d_%H%M%S")

DRIVE_TRAIN_SAVE_DIR = DRIVE_DATA_DIR / "titanet_finetune_runs" / RUN_ID
DRIVE_EVAL_SAVE_DIR = DRIVE_DATA_DIR / "titanet_eval_runs" / EVAL_RUN_ID
DRIVE_TRAIN_SAVE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_EVAL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("DRIVE_DATA_DIR:", DRIVE_DATA_DIR, DRIVE_DATA_DIR.exists())
print("DATA_ZIP_PATH:", DATA_ZIP_PATH, DATA_ZIP_PATH.exists())
print("DRIVE_TRAIN_SAVE_DIR:", DRIVE_TRAIN_SAVE_DIR)
print("DRIVE_EVAL_SAVE_DIR:", DRIVE_EVAL_SAVE_DIR)

assert DRIVE_DATA_DIR.exists(), DRIVE_DATA_DIR
assert DATA_ZIP_PATH.exists(), DATA_ZIP_PATH

DRIVE_DATA_DIR: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터 True
DATA_ZIP_PATH: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_local_subset_medium.zip True
DRIVE_TRAIN_SAVE_DIR: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_medium_5epoch_lr5e5_20260502_070940
DRIVE_EVAL_SAVE_DIR: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_eval_runs/eval_medium_5epoch_lr5e5_20260502_070940


In [ ]:
INSTALL_OR_REPAIR = False

if INSTALL_OR_REPAIR:
    !pip install -q "nemo_toolkit[asr]"
    !pip uninstall -y numpy
    !pip install --no-cache-dir --force-reinstall "numpy==1.26.4"
    print("설치/복구 완료. 런타임 재시작 후 처음부터 다시 실행하세요.")
else:
    print("Skip install/repair.")

Skip install/repair.


In [ ]:
import os, sys, json, ast, random, shutil, zipfile, gc
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import soundfile as sf
from tqdm.auto import tqdm

import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger
from omegaconf import OmegaConf, open_dict
from nemo.collections.asr.models import EncDecSpeakerLabelModel

try:
    from torchmetrics.classification import MulticlassAccuracy
except Exception:
    MulticlassAccuracy = None

print("python:", sys.version)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("lightning.pytorch:", pl.__version__)

pl.seed_everything(SEED, workers=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

[NeMo W 2026-05-02 07:09:56 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
[NeMo W 2026-05-02 07:09:58 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-05-02 07:09:58 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-05-02 07:09:58 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-05-02 07:09:58 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    
INFO: Seed set to

python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
numpy: 1.26.4
torch: 2.10.0+cu128
cuda: True
gpu: NVIDIA A100-SXM4-40GB
lightning.pytorch: 2.4.0


In [ ]:
WORK_ZIP = Path("/content/titanet_local_subset_medium.zip")
EXTRACT_ROOT = Path("/content/titanet_medium_extract")

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

if not WORK_ZIP.exists() or WORK_ZIP.stat().st_size != DATA_ZIP_PATH.stat().st_size:
    print("Copy dataset ZIP to /content ...")
    shutil.copy2(DATA_ZIP_PATH, WORK_ZIP)
else:
    print("Dataset ZIP already copied:", WORK_ZIP)

with zipfile.ZipFile(WORK_ZIP, "r") as zf:
    names = zf.namelist()
    print("ZIP first 30 entries:")
    for n in names[:30]:
        print(" -", n)
    zf.extractall(EXTRACT_ROOT)

candidates = []
if (EXTRACT_ROOT / "manifests").is_dir() and (EXTRACT_ROOT / "wavs").is_dir():
    candidates.append(EXTRACT_ROOT)
for p in EXTRACT_ROOT.rglob("*"):
    if p.is_dir() and (p / "manifests").is_dir() and (p / "wavs").is_dir():
        candidates.append(p)

candidates = sorted(set(candidates), key=lambda x: len(str(x)))
print("Dataset candidates:", candidates)
if not candidates:
    print("Extract root children:", list(EXTRACT_ROOT.iterdir()))
    raise FileNotFoundError("Could not find dataset dir with manifests/ and wavs/.")

DATA_DIR = candidates[0]
MANIFEST_DIR = DATA_DIR / "manifests"
COLAB_MANIFEST_DIR = DATA_DIR / "manifests_colab"
COLAB_MANIFEST_DIR.mkdir(exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("MANIFEST_DIR:", MANIFEST_DIR)
print("Manifest files:", [p.name for p in MANIFEST_DIR.iterdir()][:30])

Copy dataset ZIP to /content ...
ZIP first 30 entries:
 - manifests/enrollment_manifest_local.csv
 - manifests/evaluation_verification_manifest_local.csv
 - manifests/eval_manifest.json
 - manifests/train_manifest.json
 - manifests/trials_local.csv
 - manifests/valid_manifest.json
 - reports/audio_check_report.csv
 - reports/auto_shrink_report.json
 - reports/export_titanet_local_subset.log
 - reports/extracted_files.csv
 - reports/failed_extracts.csv
 - reports/speaker_distribution_report.csv
 - reports/split_leakage_report.json
 - reports/trial_distribution_report.csv
 - wavs/eval/0036/A0001-0036F1411-10000000-00009126_46676df7ac.wav
 - wavs/eval/0036/A0002-0036F1411-10000010-00010349_ac3a51cdc7.wav
 - wavs/eval/0036/A0002-0036F1411-10000040-00010325_928f24d51a.wav
 - wavs/eval/0036/A0006-0036F1411-10000010-00016739_fd4d7321e7.wav
 - wavs/eval/0036/A0016-0036F1411-10000010-00017850_1694def51f.wav
 - wavs/eval/0036/A0027-0036F1411-10000020-00066643_e6d9769310.wav
 - wavs/eval/0036/A00

In [ ]:
# =========================
# 5. JSONL utilities - fixed
# =========================
def read_json_objects_robust(path):
    """
    Handles:
    1. normal JSONL:
       {"a":1}
       {"a":2}

    2. concatenated JSON objects:
       {"a":1}{"a":2}

    3. comma-separated JSON objects:
       {"a":1},{"a":2}

    4. JSON array:
       [{"a":1}, {"a":2}]
    """
    path = Path(path)
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return []

    # Case 1: whole file is valid JSON array
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
        if isinstance(obj, dict):
            return [obj]
    except json.JSONDecodeError:
        pass

    rows = []
    decoder = json.JSONDecoder()
    idx = 0
    n = len(text)

    while idx < n:
        # skip whitespace and common separators
        while idx < n and text[idx] in " \t\r\n,[]":
            idx += 1

        if idx >= n:
            break

        # if broken separator, move to next JSON object
        if text[idx] not in "{[":
            next_obj = text.find("{", idx)
            if next_obj == -1:
                break
            idx = next_obj

        try:
            obj, next_idx = decoder.raw_decode(text, idx)
            rows.append(obj)
            idx = next_idx
        except json.JSONDecodeError as e:
            print(f"[JSON decode failed] path={path}")
            print(f"index={idx}, near text:")
            print(text[idx:idx+500])
            raise e

    return rows


def read_jsonl(path, limit=None):
    path = Path(path)
    rows = []

    # First try strict JSONL
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue

                # tolerate trailing comma in each line
                if line.endswith(","):
                    line = line[:-1].strip()

                # skip JSON array brackets if accidentally present
                if line in ["[", "]"]:
                    continue

                rows.append(json.loads(line))

                if limit is not None and len(rows) >= limit:
                    break

        return rows

    except json.JSONDecodeError:
        print(f"[WARN] Strict JSONL failed. Trying robust parser: {path}")
        rows = read_json_objects_robust(path)
        if limit is not None:
            rows = rows[:limit]
        return rows


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


print("JSONL utilities fixed and ready.")

JSONL utilities fixed and ready.


In [ ]:
# =========================
# Repair: create train_manifest_colab.json if missing
# =========================
from pathlib import Path
import json

print("DATA_DIR:", DATA_DIR)
print("MANIFEST_DIR:", MANIFEST_DIR)
print("COLAB_MANIFEST_DIR:", COLAB_MANIFEST_DIR)

COLAB_MANIFEST_DIR.mkdir(exist_ok=True)

LOCAL_ROOT_MARKERS = [
    "D:/aihub_check/titanet_local_subset_medium",
    r"D:\aihub_check\titanet_local_subset_medium",
]
COLAB_ROOT_STR = str(DATA_DIR).replace("\\", "/")


def rewrite_path_to_colab(path_str):
    if path_str is None:
        return path_str

    s = str(path_str).replace("\\", "/")

    for marker in LOCAL_ROOT_MARKERS:
        m = marker.replace("\\", "/")
        if s.startswith(m):
            return s.replace(m, COLAB_ROOT_STR, 1)

    if s.startswith("/content/"):
        return s

    idx = s.find("/wavs/")
    if idx >= 0:
        return COLAB_ROOT_STR + s[idx:]

    return s


def read_json_objects_robust(path):
    path = Path(path)
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return []

    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
        if isinstance(obj, dict):
            return [obj]
    except json.JSONDecodeError:
        pass

    rows = []
    decoder = json.JSONDecoder()
    idx = 0
    n = len(text)

    while idx < n:
        while idx < n and text[idx] in " \t\r\n,[]":
            idx += 1

        if idx >= n:
            break

        if text[idx] not in "{[":
            next_obj = text.find("{", idx)
            if next_obj == -1:
                break
            idx = next_obj

        obj, next_idx = decoder.raw_decode(text, idx)
        rows.append(obj)
        idx = next_idx

    return rows


def read_jsonl(path, limit=None):
    path = Path(path)
    rows = []

    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                if line.endswith(","):
                    line = line[:-1].strip()

                if line in ["[", "]"]:
                    continue

                rows.append(json.loads(line))

                if limit is not None and len(rows) >= limit:
                    break

        return rows

    except json.JSONDecodeError:
        rows = read_json_objects_robust(path)
        return rows[:limit] if limit is not None else rows


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def rewrite_jsonl_manifest(src, dst):
    rows = read_jsonl(src)
    missing = 0

    for row in rows:
        row["audio_filepath"] = rewrite_path_to_colab(row.get("audio_filepath"))

        if not Path(row["audio_filepath"]).exists():
            missing += 1

    write_jsonl(dst, rows)

    return {
        "src": str(src),
        "dst": str(dst),
        "rows": len(rows),
        "missing_paths": missing,
    }


rewrite_reports = []

for src_name, dst_name in [
    ("train_manifest.json", "train_manifest_colab.json"),
    ("valid_manifest.json", "valid_manifest_colab.json"),
    ("eval_manifest.json", "eval_manifest_colab.json"),
]:
    src = MANIFEST_DIR / src_name
    dst = COLAB_MANIFEST_DIR / dst_name

    assert src.exists(), f"missing source manifest: {src}"

    report = rewrite_jsonl_manifest(src, dst)
    rewrite_reports.append(report)

print(json.dumps(rewrite_reports, ensure_ascii=False, indent=2))

for report in rewrite_reports:
    assert report["missing_paths"] == 0, report

SOURCE_TRAIN_MANIFEST = COLAB_MANIFEST_DIR / "train_manifest_colab.json"
assert SOURCE_TRAIN_MANIFEST.exists(), SOURCE_TRAIN_MANIFEST

print("OK:", SOURCE_TRAIN_MANIFEST)

DATA_DIR: /content/titanet_medium_extract
MANIFEST_DIR: /content/titanet_medium_extract/manifests
COLAB_MANIFEST_DIR: /content/titanet_medium_extract/manifests_colab
[
  {
    "src": "/content/titanet_medium_extract/manifests/train_manifest.json",
    "dst": "/content/titanet_medium_extract/manifests_colab/train_manifest_colab.json",
    "rows": 72000,
    "missing_paths": 0
  },
  {
    "src": "/content/titanet_medium_extract/manifests/valid_manifest.json",
    "dst": "/content/titanet_medium_extract/manifests_colab/valid_manifest_colab.json",
    "rows": 8400,
    "missing_paths": 0
  },
  {
    "src": "/content/titanet_medium_extract/manifests/eval_manifest.json",
    "dst": "/content/titanet_medium_extract/manifests_colab/eval_manifest_colab.json",
    "rows": 5000,
    "missing_paths": 0
  }
]
OK: /content/titanet_medium_extract/manifests_colab/train_manifest_colab.json


In [ ]:
SOURCE_TRAIN_MANIFEST = COLAB_MANIFEST_DIR / "train_manifest_colab.json"
TRAIN_FT_MANIFEST = COLAB_MANIFEST_DIR / "train_ft_manifest.json"
VAL_FT_MANIFEST = COLAB_MANIFEST_DIR / "val_ft_manifest.json"

rows = read_jsonl(SOURCE_TRAIN_MANIFEST)
if LIMIT_TRAIN_ROWS and LIMIT_TRAIN_ROWS > 0:
    by_label_tmp = defaultdict(list)
    for r in rows:
        by_label_tmp[str(r["label"])].append(r)
    rng = random.Random(SEED)
    labels_tmp = sorted(by_label_tmp)
    for lab in labels_tmp:
        rng.shuffle(by_label_tmp[lab])
    sampled, exhausted = [], False
    while len(sampled) < LIMIT_TRAIN_ROWS and not exhausted:
        exhausted = True
        for lab in labels_tmp:
            if by_label_tmp[lab] and len(sampled) < LIMIT_TRAIN_ROWS:
                sampled.append(by_label_tmp[lab].pop())
                exhausted = False
    rows = sampled

rows_by_label = defaultdict(list)
for r in rows:
    rows_by_label[str(r["label"])].append(r)

train_rows, val_rows = [], []
rng = random.Random(SEED)
for label, lab_rows in rows_by_label.items():
    rng.shuffle(lab_rows)
    n = len(lab_rows)
    n_val = max(1, int(n * 0.1)) if n >= 10 else (1 if n >= 3 else 0)
    val_rows.extend(lab_rows[:n_val])
    train_rows.extend(lab_rows[n_val:])

if LIMIT_VALID_ROWS and LIMIT_VALID_ROWS > 0:
    val_rows = val_rows[:LIMIT_VALID_ROWS]

write_jsonl(TRAIN_FT_MANIFEST, train_rows)
write_jsonl(VAL_FT_MANIFEST, val_rows)

train_labels = set(str(r["label"]) for r in train_rows)
val_labels = set(str(r["label"]) for r in val_rows)
summary = {
    "source_train_rows_after_limit": len(rows),
    "train_ft_rows": len(train_rows),
    "val_ft_rows": len(val_rows),
    "train_ft_speakers": len(train_labels),
    "val_ft_speakers": len(val_labels),
    "val_labels_not_in_train": len(val_labels - train_labels),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
(DRIVE_TRAIN_SAVE_DIR / "ft_split_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
assert len(val_labels - train_labels) == 0
assert len(train_rows) > 0 and len(val_rows) > 0

{
  "source_train_rows_after_limit": 72000,
  "train_ft_rows": 64800,
  "val_ft_rows": 7200,
  "train_ft_speakers": 900,
  "val_ft_speakers": 900,
  "val_labels_not_in_train": 0
}


In [ ]:
def validate_manifest_audio(manifest_path, sample_count=200):
    rows = read_jsonl(manifest_path, limit=sample_count)
    ok, failed, failed_items = 0, 0, []
    for i, row in enumerate(tqdm(rows, desc=f"validate {Path(manifest_path).name}")):
        audio_path = row.get("audio_filepath")
        label = row.get("label")
        try:
            p = Path(audio_path)
            if not p.exists():
                raise FileNotFoundError(str(p))
            info = sf.info(str(p))
            if info.samplerate != 16000:
                raise ValueError(f"bad sample_rate={info.samplerate}")
            if info.channels != 1:
                raise ValueError(f"bad channels={info.channels}")
            if info.duration <= 0:
                raise ValueError(f"bad duration={info.duration}")
            if label is None or str(label) == "":
                raise ValueError("missing label")
            ok += 1
        except Exception as e:
            failed += 1
            failed_items.append({"index": i, "audio_filepath": audio_path, "label": label, "error": repr(e)})
    report = {"manifest": str(manifest_path), "checked": len(rows), "ok": ok, "failed": failed, "failed_items_sample": failed_items[:20]}
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return report

audio_reports = [validate_manifest_audio(TRAIN_FT_MANIFEST, 200), validate_manifest_audio(VAL_FT_MANIFEST, 200)]
(DRIVE_TRAIN_SAVE_DIR / "manifest_audio_validation_report.json").write_text(
    json.dumps(audio_reports, ensure_ascii=False, indent=2), encoding="utf-8"
)
assert all(r["failed"] == 0 for r in audio_reports)

validate train_ft_manifest.json:   0%|          | 0/200 [00:00<?, ?it/s]

{
  "manifest": "/content/titanet_medium_extract/manifests_colab/train_ft_manifest.json",
  "checked": 200,
  "ok": 200,
  "failed": 0,
  "failed_items_sample": []
}


validate val_ft_manifest.json:   0%|          | 0/200 [00:00<?, ?it/s]

{
  "manifest": "/content/titanet_medium_extract/manifests_colab/val_ft_manifest.json",
  "checked": 200,
  "ok": 200,
  "failed": 0,
  "failed_items_sample": []
}


In [ ]:
def get_train_labels(manifest_path):
    return sorted({str(r["label"]) for r in read_jsonl(manifest_path)})

def replace_last_linear_in_module(module, out_features):
    linear_names = [name for name, child in module.named_modules() if isinstance(child, nn.Linear)]
    if not linear_names:
        return False
    target_name = linear_names[-1]
    parent = module
    parts = target_name.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    old = getattr(parent, parts[-1])
    new = nn.Linear(old.in_features, out_features, bias=old.bias is not None)
    new.to(device=old.weight.device, dtype=old.weight.dtype)
    setattr(parent, parts[-1], new)
    print(f"Replaced decoder linear: {target_name}, {old.out_features} -> {out_features}")
    return True

def reset_classification_metrics(model, num_classes):
    device = next(model.parameters()).device
    if MulticlassAccuracy is not None:
        for name in ["_macro_accuracy", "_pair_macro_accuracy"]:
            if hasattr(model, name):
                try:
                    setattr(model, name, MulticlassAccuracy(num_classes=num_classes, average="macro").to(device))
                    print(f"Reset metric: {name} -> num_classes={num_classes}")
                except Exception as e:
                    print(f"[WARN] failed to reset {name}: {e}")

def configure_speaker_model_for_finetune(model, train_manifest, val_manifest, labels):
    num_classes = len(labels)
    with open_dict(model.cfg):
        model.cfg.labels = labels
        if "decoder" in model.cfg:
            for k in ["num_classes", "num_classes_total", "n_classes", "num_speakers"]:
                if k in model.cfg.decoder:
                    model.cfg.decoder[k] = num_classes
        if "optim" in model.cfg and model.cfg.optim is not None:
            model.cfg.optim.lr = LEARNING_RATE

    for attr in ["labels", "_labels"]:
        try:
            setattr(model, attr, labels)
        except Exception:
            pass

    replace_last_linear_in_module(model.decoder, num_classes)
    reset_classification_metrics(model, num_classes)

    train_cfg = OmegaConf.create({
        "manifest_filepath": str(train_manifest),
        "sample_rate": 16000,
        "labels": labels,
        "batch_size": BATCH_SIZE,
        "shuffle": True,
        "is_tarred": False,
        "num_workers": NUM_WORKERS,
        "pin_memory": True,
        "min_duration": MIN_AUDIO_DURATION,
        "max_duration": MAX_AUDIO_DURATION,
    })

    val_cfg = OmegaConf.create({
        "manifest_filepath": str(val_manifest),
        "sample_rate": 16000,
        "labels": labels,
        "batch_size": BATCH_SIZE,
        "shuffle": False,
        "is_tarred": False,
        "num_workers": NUM_WORKERS,
        "pin_memory": True,
        "min_duration": MIN_AUDIO_DURATION,
        "max_duration": MAX_AUDIO_DURATION,
    })
    model.setup_training_data(train_cfg)
    model.setup_validation_data(val_cfg)
    return model

def make_trainer(model_run_dir):
    model_run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_cb = ModelCheckpoint(
        dirpath=str(model_run_dir / "checkpoints"),
        filename="{epoch}-{step}-{val_loss:.4f}",
        monitor="val_loss",
        mode="min",
        save_top_k=2,
        save_last=True,
    )
    lr_cb = LearningRateMonitor(logging_interval="step")
    logger = CSVLogger(save_dir=str(model_run_dir), name="logs")
    return pl.Trainer(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        precision=PRECISION if torch.cuda.is_available() else "32-true",
        max_epochs=MAX_EPOCHS,
        limit_train_batches=LIMIT_TRAIN_BATCHES,
        limit_val_batches=LIMIT_VAL_BATCHES,
        num_sanity_val_steps=0,
        log_every_n_steps=10,
        callbacks=[ckpt_cb, lr_cb],
        logger=logger,
        default_root_dir=str(model_run_dir),
        enable_checkpointing=True,
    )

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

labels = get_train_labels(TRAIN_FT_MANIFEST)
print("num train labels:", len(labels), labels[:10])

model = EncDecSpeakerLabelModel.from_pretrained(BASELINE_MODEL_NAME)
print("Loaded:", BASELINE_MODEL_NAME)
print("LightningModule:", isinstance(model, pl.LightningModule))

model = configure_speaker_model_for_finetune(model, TRAIN_FT_MANIFEST, VAL_FT_MANIFEST, labels)

model_run_dir = DRIVE_TRAIN_SAVE_DIR / "titanet_small"
trainer = make_trainer(model_run_dir)

run_config = {
    "run_id": RUN_ID,
    "model_name": BASELINE_MODEL_NAME,
    "dataset": "titanet_local_subset_medium",
    "max_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "precision": PRECISION,
    "train_manifest": str(TRAIN_FT_MANIFEST),
    "val_manifest": str(VAL_FT_MANIFEST),
    "num_labels": len(labels),
    "drive_data_dir": str(DRIVE_DATA_DIR),
}
model_run_dir.mkdir(parents=True, exist_ok=True)
(model_run_dir / "run_config.json").write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

print("Start fine-tuning")
print(json.dumps(run_config, ensure_ascii=False, indent=2))
trainer.fit(model)

FINETUNED_NEMO_PATH = model_run_dir / "titanet_small_finetuned_final.nemo"
model.save_to(str(FINETUNED_NEMO_PATH))

(model_run_dir / "labels.json").write_text(json.dumps(labels, ensure_ascii=False, indent=2), encoding="utf-8")
OmegaConf.save(model.cfg, model_run_dir / "model_cfg.yaml")
train_meta = {**run_config, "nemo_path": str(FINETUNED_NEMO_PATH), "model_run_dir": str(model_run_dir)}
(model_run_dir / "train_meta.json").write_text(json.dumps(train_meta, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:", FINETUNED_NEMO_PATH)
print("exists:", FINETUNED_NEMO_PATH.exists())
print("size MB:", FINETUNED_NEMO_PATH.stat().st_size / 1024 / 1024)

restored = EncDecSpeakerLabelModel.restore_from(str(FINETUNED_NEMO_PATH))
print("restore OK:", type(restored))
del restored
torch.cuda.empty_cache()
gc.collect()

num train labels: 900 ['0024', '0102', '0210', '0271', '0318', '0329', '0330', '0342', '0343', '0345']
[NeMo I 2026-05-02 07:11:29 cloud:68] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/titanet_small/versions/1.19.0/files/titanet-s.nemo to /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo
[NeMo I 2026-05-02 07:11:30 common:939] Instantiating model from pre-trained checkpoint


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
    - id07182
    - id07183
    - id07185
    - id07186
    - id07187
    - id07188
    - id07189
    - id07191
    - id07192
    - id07194
    - id07195
    - id07196
    - id07197
    - id07198
    - id07199
    - id07200
    - id07202
    - id07204
    - id07205
    - id07206
    - id07207
    - id07208
    - id07209
    - id07210
    - id07212
    - id07213
    - id07214
    - id07215
    - id07217
    - id07218
    - id07219
    - id07220
    - id07221
    - id07223
    - id07227
    - id07228
    - id07229
    - id07230
    - id07232
    - id07233
    - id07234
    - id07235
    - id07236
    - id07238
    - id07240
    - id07241
    - id07242
    - id07243
    - id07244
    - id07246
    - id07247
    - id07250
    - id07251
    - id07253
    - id07254
    - id07255
    - id07256
    - id07258
    - id07259
    - id07262
    - id07263
    - id07264
    - id07265
    - id07268
    - id07269
    - id07272
    - id07273
    - id07275
    - id0727

[NeMo I 2026-05-02 07:11:36 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo.
Loaded: titanet_small
LightningModule: True
Replaced decoder linear: final, 16681 -> 900
Reset metric: _macro_accuracy -> num_classes=900
Reset metric: _pair_macro_accuracy -> num_classes=900
[NeMo I 2026-05-02 07:11:37 collections:750] Filtered duration for loading collection is  0.08 hours.
[NeMo I 2026-05-02 07:11:37 collections:751] Dataset successfully loaded with 64789 items and total duration provided from manifest is  35.82 hours.
[NeMo I 2026-05-02 07:11:37 collections:757] # 64789 files loaded accounting to # 900 labels


[NeMo W 2026-05-02 07:11:37 label_models:201] Total number of 900 labels found in all the manifest files.


[NeMo I 2026-05-02 07:11:37 collections:750] Filtered duration for loading collection is  0.08 hours.
[NeMo I 2026-05-02 07:11:37 collections:751] Dataset successfully loaded with 64789 items and total duration provided from manifest is  35.82 hours.
[NeMo I 2026-05-02 07:11:37 collections:757] # 64789 files loaded accounting to # 900 labels
[NeMo I 2026-05-02 07:11:37 collections:750] Filtered duration for loading collection is  0.01 hours.
[NeMo I 2026-05-02 07:11:37 collections:751] Dataset successfully loaded with 7197 items and total duration provided from manifest is  3.96 hours.
[NeMo I 2026-05-02 07:11:37 collections:757] # 7197 files loaded accounting to # 900 labels


...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


Saved: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_medium_5epoch_lr5e5_20260502_070940/titanet_small/titanet_small_finetuned_final.nemo
exists: True
size MB: 27.12890625


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
    - id07182
    - id07183
    - id07185
    - id07186
    - id07187
    - id07188
    - id07189
    - id07191
    - id07192
    - id07194
    - id07195
    - id07196
    - id07197
    - id07198
    - id07199
    - id07200
    - id07202
    - id07204
    - id07205
    - id07206
    - id07207
    - id07208
    - id07209
    - id07210
    - id07212
    - id07213
    - id07214
    - id07215
    - id07217
    - id07218
    - id07219
    - id07220
    - id07221
    - id07223
    - id07227
    - id07228
    - id07229
    - id07230
    - id07232
    - id07233
    - id07234
    - id07235
    - id07236
    - id07238
    - id07240
    - id07241
    - id07242
    - id07243
    - id07244
    - id07246
    - id07247
    - id07250
    - id07251
    - id07253
    - id07254
    - id07255
    - id07256
    - id07258
    - id07259
    - id07262
    - id07263
    - id07264
    - id07265
    - id07268
    - id07269
    - id07272
    - id07273
    - id07275
    - id0727

[NeMo I 2026-05-02 08:25:44 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_medium_5epoch_lr5e5_20260502_070940/titanet_small/titanet_small_finetuned_final.nemo.
restore OK: <class 'nemo.collections.asr.models.label_models.EncDecSpeakerLabelModel'>


429315

In [ ]:
if not RUN_EVAL_AFTER_TRAIN:
    raise SystemExit("RUN_EVAL_AFTER_TRAIN=False. Training finished.")

def parse_audio_refs(x):
    if pd.isna(x):
        return []
    s = str(x)
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, list):
            return [str(v) for v in obj]
    except Exception:
        pass
    if ";" in s:
        return [v for v in s.split(";") if v]
    return [s]

def rewrite_audio_refs_field(x):
    return str([rewrite_path_to_colab(v) for v in parse_audio_refs(x)])

def rewrite_trials_to_colab(src, dst):
    df = pd.read_csv(src)
    required = ["verification_audio_ref", "enrollment_audio_refs", "enroll_sec", "label"]
    missing = [c for c in required if c not in df.columns]
    assert not missing, f"Missing trial columns: {missing}"
    df["verification_audio_ref"] = df["verification_audio_ref"].map(rewrite_path_to_colab)
    df["enrollment_audio_refs"] = df["enrollment_audio_refs"].map(rewrite_audio_refs_field)
    df.to_csv(dst, index=False)
    return df

trial_candidates = [MANIFEST_DIR / "trials_local.csv", MANIFEST_DIR / "trials_colab.csv", COLAB_MANIFEST_DIR / "trials_colab.csv"]
trials_src = next((p for p in trial_candidates if p.exists()), None)
assert trials_src is not None, f"No trials file found in {trial_candidates}"

TRIALS_COLAB = COLAB_MANIFEST_DIR / "trials_colab.csv"
trials_df = rewrite_trials_to_colab(trials_src, TRIALS_COLAB)
print("trials rows:", len(trials_df))
print(trials_df.groupby(["enroll_sec", "label"]).size())

df = pd.read_csv(TRIALS_COLAB)
df["enroll_sec"] = df["enroll_sec"].astype(float)
df["label"] = df["label"].astype(int)

if MAX_TRIALS_PER_ENROLL_SEC and MAX_TRIALS_PER_ENROLL_SEC > 0:
    sampled_parts = []
    for enroll_sec, g in df.groupby("enroll_sec"):
        n_total = min(len(g), MAX_TRIALS_PER_ENROLL_SEC)
        label_ratio = g["label"].value_counts(normalize=True).to_dict()
        sub_parts, used = [], 0
        for label_value in sorted(g["label"].unique()):
            gg = g[g["label"] == label_value]
            n = min(int(round(n_total * label_ratio.get(label_value, 0))), len(gg))
            if n > 0:
                sub_parts.append(gg.sample(n=n, random_state=SEED + int(enroll_sec * 10) + label_value))
                used += n
        if used < n_total:
            used_idx = pd.concat(sub_parts).index if sub_parts else []
            rest = g.drop(index=used_idx, errors="ignore")
            if len(rest) > 0:
                sub_parts.append(rest.sample(n=min(n_total-used, len(rest)), random_state=SEED))
        sampled_parts.append(pd.concat(sub_parts))
    eval_trials = pd.concat(sampled_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
else:
    eval_trials = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

EVAL_TRIALS_PATH = DRIVE_EVAL_SAVE_DIR / "eval_trials_used.csv"
eval_trials.to_csv(EVAL_TRIALS_PATH, index=False)
print("eval_trials:", len(eval_trials))
print(eval_trials.groupby(["enroll_sec", "label"]).size())

trials rows: 80000
enroll_sec  label
1.5         0        10000
            1         3337
2.0         0        10000
            1         3340
2.5         0        10000
            1         3326
3.0         0        10000
            1         3327
4.0         0        10000
            1         3331
5.0         0        10000
            1         3339
dtype: int64
eval_trials: 80000
enroll_sec  label
1.5         0        10000
            1         3337
2.0         0        10000
            1         3340
2.5         0        10000
            1         3326
3.0         0        10000
            1         3327
4.0         0        10000
            1         3331
5.0         0        10000
            1         3339
dtype: int64


In [ ]:
if "model" in globals():
    del model
torch.cuda.empty_cache()
gc.collect()

def load_baseline():
    m = EncDecSpeakerLabelModel.from_pretrained(BASELINE_MODEL_NAME)
    m.eval()
    m.to(device)
    return m

def load_finetuned():
    m = EncDecSpeakerLabelModel.restore_from(str(FINETUNED_NEMO_PATH))
    m.eval()
    m.to(device)
    return m

baseline_model = load_baseline()
finetuned_model = load_finetuned()

def extract_embedding(model, audio_path):
    audio_path = str(audio_path)
    last_err = None
    if hasattr(model, "get_embedding"):
        try:
            emb = model.get_embedding(audio_path)
            if isinstance(emb, torch.Tensor):
                emb = emb.detach().cpu().float().numpy()
            return np.asarray(emb).squeeze()
        except Exception as e:
            last_err = e
    if hasattr(model, "infer_file"):
        try:
            out = model.infer_file(audio_path)
            if isinstance(out, tuple):
                for item in out:
                    arr = np.asarray(item)
                    if arr.size > 10:
                        return arr.squeeze()
            return np.asarray(out).squeeze()
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Could not extract embedding. path={audio_path}, err={repr(last_err)}")

def normalize_embedding(x):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    return x / (np.linalg.norm(x) + 1e-12)

def collect_unique_audio_paths(trials):
    paths = set()
    for _, row in trials.iterrows():
        paths.add(rewrite_path_to_colab(row["verification_audio_ref"]))
        for ref in parse_audio_refs(row["enrollment_audio_refs"]):
            paths.add(rewrite_path_to_colab(ref))
    paths = sorted(paths)
    missing = [p for p in paths if not Path(p).exists()]
    if missing:
        print("Missing sample:", missing[:20])
        raise FileNotFoundError(f"Missing {len(missing)} audio files")
    return paths

unique_audio_paths = collect_unique_audio_paths(eval_trials)
print("unique_audio_paths:", len(unique_audio_paths))

def extract_embeddings_for_model(model, model_tag, audio_paths):
    cache, failures = {}, []
    for p in tqdm(audio_paths, desc=f"extract {model_tag}"):
        try:
            emb = extract_embedding(model, p)
            cache[p] = normalize_embedding(emb)
        except Exception as e:
            failures.append({"audio_path": p, "error": repr(e)})
    fail_path = DRIVE_EVAL_SAVE_DIR / f"{model_tag}_embedding_failures.csv"
    pd.DataFrame(failures).to_csv(fail_path, index=False)
    print(model_tag, "success:", len(cache), "failures:", len(failures))
    if not cache:
        raise RuntimeError(f"No embeddings extracted for {model_tag}")
    return cache, failures

baseline_embs, baseline_failures = extract_embeddings_for_model(baseline_model, "baseline", unique_audio_paths)
finetuned_embs, finetuned_failures = extract_embeddings_for_model(finetuned_model, "finetuned_medium_5epoch_lr5e5", unique_audio_paths)

[NeMo I 2026-05-02 08:33:25 cloud:58] Found existing object /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo.
[NeMo I 2026-05-02 08:33:25 cloud:64] Re-using file from: /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo
[NeMo I 2026-05-02 08:33:25 common:939] Instantiating model from pre-trained checkpoint


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
    - id07182
    - id07183
    - id07185
    - id07186
    - id07187
    - id07188
    - id07189
    - id07191
    - id07192
    - id07194
    - id07195
    - id07196
    - id07197
    - id07198
    - id07199
    - id07200
    - id07202
    - id07204
    - id07205
    - id07206
    - id07207
    - id07208
    - id07209
    - id07210
    - id07212
    - id07213
    - id07214
    - id07215
    - id07217
    - id07218
    - id07219
    - id07220
    - id07221
    - id07223
    - id07227
    - id07228
    - id07229
    - id07230
    - id07232
    - id07233
    - id07234
    - id07235
    - id07236
    - id07238
    - id07240
    - id07241
    - id07242
    - id07243
    - id07244
    - id07246
    - id07247
    - id07250
    - id07251
    - id07253
    - id07254
    - id07255
    - id07256
    - id07258
    - id07259
    - id07262
    - id07263
    - id07264
    - id07265
    - id07268
    - id07269
    - id07272
    - id07273
    - id07275
    - id0727

[NeMo I 2026-05-02 08:33:30 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.3/titanet-s/908e9576cf7dd7420e75f73ceb0b72e1/titanet-s.nemo.


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
    - id07182
    - id07183
    - id07185
    - id07186
    - id07187
    - id07188
    - id07189
    - id07191
    - id07192
    - id07194
    - id07195
    - id07196
    - id07197
    - id07198
    - id07199
    - id07200
    - id07202
    - id07204
    - id07205
    - id07206
    - id07207
    - id07208
    - id07209
    - id07210
    - id07212
    - id07213
    - id07214
    - id07215
    - id07217
    - id07218
    - id07219
    - id07220
    - id07221
    - id07223
    - id07227
    - id07228
    - id07229
    - id07230
    - id07232
    - id07233
    - id07234
    - id07235
    - id07236
    - id07238
    - id07240
    - id07241
    - id07242
    - id07243
    - id07244
    - id07246
    - id07247
    - id07250
    - id07251
    - id07253
    - id07254
    - id07255
    - id07256
    - id07258
    - id07259
    - id07262
    - id07263
    - id07264
    - id07265
    - id07268
    - id07269
    - id07272
    - id07273
    - id07275
    - id0727

[NeMo I 2026-05-02 08:33:36 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_medium_5epoch_lr5e5_20260502_070940/titanet_small/titanet_small_finetuned_final.nemo.
unique_audio_paths: 5363


extract baseline:   0%|          | 0/5363 [00:00<?, ?it/s]

baseline success: 5363 failures: 0


extract finetuned_medium_5epoch_lr5e5:   0%|          | 0/5363 [00:00<?, ?it/s]

finetuned_medium_5epoch_lr5e5 success: 5363 failures: 0


In [ ]:
def score_trials(trials, emb_cache):
    rows, skipped = [], 0
    for _, row in tqdm(trials.iterrows(), total=len(trials), desc="score trials"):
        verification_path = rewrite_path_to_colab(row["verification_audio_ref"])
        enroll_paths = [rewrite_path_to_colab(x) for x in parse_audio_refs(row["enrollment_audio_refs"])]
        if verification_path not in emb_cache:
            skipped += 1
            continue
        enroll_vecs = [emb_cache[p] for p in enroll_paths if p in emb_cache]
        if not enroll_vecs:
            skipped += 1
            continue
        enroll_emb = normalize_embedding(np.mean(np.stack(enroll_vecs, axis=0), axis=0))
        ver_emb = emb_cache[verification_path]
        score = float(np.dot(enroll_emb, ver_emb))
        rows.append({
            "trial_id": row.get("trial_id", ""),
            "split": row.get("split", ""),
            "enroll_sec": float(row["enroll_sec"]),
            "label": int(row["label"]),
            "trial_type": row.get("trial_type", ""),
            "score": score,
            "enroll_speaker_id": row.get("enroll_speaker_id", ""),
            "test_speaker_id": row.get("test_speaker_id", ""),
        })
    return pd.DataFrame(rows), skipped

baseline_scores, baseline_skipped = score_trials(eval_trials, baseline_embs)
finetuned_scores, finetuned_skipped = score_trials(eval_trials, finetuned_embs)
baseline_scores["model"] = "baseline"
finetuned_scores["model"] = "finetuned_medium_5epoch_lr5e5"
baseline_scores.to_csv(DRIVE_EVAL_SAVE_DIR / "baseline_trial_scores.csv", index=False)
finetuned_scores.to_csv(DRIVE_EVAL_SAVE_DIR / "finetuned_medium_5epoch_lr5e5_trial_scores.csv", index=False)
print("baseline_scores:", baseline_scores.shape, "skipped:", baseline_skipped)
print("finetuned_scores:", finetuned_scores.shape, "skipped:", finetuned_skipped)

score trials:   0%|          | 0/80000 [00:00<?, ?it/s]

score trials:   0%|          | 0/80000 [00:00<?, ?it/s]

baseline_scores: (80000, 9) skipped: 0
finetuned_scores: (80000, 9) skipped: 0


In [ ]:
def compute_eer(labels, scores):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)
    thresholds = np.unique(scores)
    thresholds = np.concatenate([[scores.max() + 1e-6], thresholds[::-1], [scores.min() - 1e-6]])
    pos = labels == 1
    neg = labels == 0
    n_pos = max(pos.sum(), 1)
    n_neg = max(neg.sum(), 1)
    fars, frrs = [], []
    for th in thresholds:
        pred_pos = scores >= th
        fars.append(np.logical_and(pred_pos, neg).sum() / n_neg)
        frrs.append(np.logical_and(~pred_pos, pos).sum() / n_pos)
    fars = np.asarray(fars)
    frrs = np.asarray(frrs)
    idx = int(np.argmin(np.abs(fars - frrs)))
    return float((fars[idx] + frrs[idx]) / 2.0), float(thresholds[idx])

def tar_at_far(labels, scores, target_far):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)
    thresholds = np.unique(scores)
    thresholds = np.concatenate([[scores.max() + 1e-6], thresholds[::-1], [scores.min() - 1e-6]])
    pos = labels == 1
    neg = labels == 0
    n_pos = max(pos.sum(), 1)
    n_neg = max(neg.sum(), 1)
    best_tar, best_far, best_th = 0.0, 0.0, float(thresholds[0])
    for th in thresholds:
        pred_pos = scores >= th
        far = np.logical_and(pred_pos, neg).sum() / n_neg
        tar = np.logical_and(pred_pos, pos).sum() / n_pos
        if far <= target_far and tar >= best_tar:
            best_tar, best_far, best_th = float(tar), float(far), float(th)
    return best_tar, best_far, best_th

def summarize_scores(df, model_name):
    results = []
    for enroll_sec, g in df.groupby("enroll_sec"):
        y = g["label"].to_numpy()
        s = g["score"].to_numpy()
        eer, eer_th = compute_eer(y, s)
        row = {
            "model": model_name,
            "enroll_sec": float(enroll_sec),
            "n_trials": int(len(g)),
            "n_pos": int((y == 1).sum()),
            "n_neg": int((y == 0).sum()),
            "eer": eer,
            "eer_threshold": eer_th,
            "pos_score_mean": float(s[y == 1].mean()) if (y == 1).any() else None,
            "neg_score_mean": float(s[y == 0].mean()) if (y == 0).any() else None,
        }
        for tfar in TARGET_FARS:
            tar, far, th = tar_at_far(y, s, tfar)
            row[f"tar_at_far_{tfar}"] = tar
            row[f"actual_far_at_far_{tfar}"] = far
            row[f"threshold_at_far_{tfar}"] = th
        results.append(row)
    return pd.DataFrame(results)

baseline_summary = summarize_scores(baseline_scores, "baseline")
finetuned_summary = summarize_scores(finetuned_scores, "finetuned_medium_5epoch_lr5e5")
summary = pd.concat([baseline_summary, finetuned_summary], ignore_index=True)
summary_path = DRIVE_EVAL_SAVE_DIR / "verification_summary_by_enroll_sec.csv"
summary.to_csv(summary_path, index=False)
display(summary)
print("saved:", summary_path)

,model,enroll_sec,n_trials,n_pos,n_neg,eer,eer_threshold,pos_score_mean,neg_score_mean,tar_at_far_0.01,actual_far_at_far_0.01,threshold_at_far_0.01,tar_at_far_0.05,actual_far_at_far_0.05,threshold_at_far_0.05,tar_at_far_0.1,actual_far_at_far_0.1,threshold_at_far_0.1
0,baseline,1.5,13337,3337,10000,0.220879,0.298211,0.396571,0.214730,0.239137,0.01,0.488099,0.473479,0.05,0.405717,0.613126,0.1,0.361377
1,baseline,2.0,13340,3340,10000,0.220979,0.296380,0.395308,0.213193,0.225150,0.01,0.491577,0.460778,0.05,0.409208,0.606587,0.1,0.360951
2,baseline,2.5,13326,3326,10000,0.219792,0.301475,0.403408,0.216840,0.245340,0.01,0.495315,0.484666,0.05,0.412121,0.616657,0.1,0.366612
3,baseline,3.0,13327,3327,10000,0.214654,0.314651,0.416314,0.227214,0.231440,0.01,0.512109,0.491133,0.05,0.422722,0.630598,0.1,0.375993
4,baseline,4.0,13331,3331,10000,0.203221,0.331024,0.438564,0.238183,0.286100,0.01,0.511701,0.549385,0.05,0.427542,0.673972,0.1,0.383656
5,baseline,5.0,13339,3339,10000,0.189939,0.347388,0.455135,0.246949,0.277628,0.01,0.532195,0.527403,0.05,0.447839,0.676550,0.1,0.400087
6,finetuned_medium_5epoch_lr5e5,1.5,13337,3337,10000,0.181300,0.284130,0.475262,0.053444,0.261912,0.01,0.627170,0.534312,0.05,0.494629,0.698232,0.1,0.397241
7,finetuned_medium_5epoch_lr5e5,2.0,13340,3340,10000,0.181718,0.281556,0.475385,0.055415,0.242216,0.01,0.637443,0.542814,0.05,0.491049,0.703293,0.1,0.394126
8,finetuned_medium_5epoch_lr5e5,2.5,13326,3326,10000,0.183402,0.285823,0.481379,0.059759,0.255262,0.01,0.637309,0.552014,0.05,0.496395,0.705352,0.1,0.396275
9,finetuned_medium_5epoch_lr5e5,3.0,13327,3327,10000,0.176117,0.297168,0.495250,0.058495,0.263601,0.01,0.646936,0.546138,0.05,0.512338,0.709047,0.1,0.409399


saved: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_eval_runs/eval_medium_5epoch_lr5e5_20260502_070940/verification_summary_by_enroll_sec.csv


In [ ]:
comparison_rows = []
for enroll_sec in sorted(summary["enroll_sec"].unique()):
    b = summary[(summary["model"] == "baseline") & (summary["enroll_sec"] == enroll_sec)].iloc[0]
    f = summary[(summary["model"] == "finetuned_medium_5epoch_lr5e5") & (summary["enroll_sec"] == enroll_sec)].iloc[0]
    row = {
        "enroll_sec": float(enroll_sec),
        "baseline_eer": float(b["eer"]),
        "finetuned_medium_5epoch_lr5e5_eer": float(f["eer"]),
        "eer_delta_finetuned_minus_baseline": float(f["eer"] - b["eer"]),
        "eer_improved": bool(f["eer"] < b["eer"]),
        "baseline_pos_score_mean": float(b["pos_score_mean"]),
        "finetuned_pos_score_mean": float(f["pos_score_mean"]),
        "baseline_neg_score_mean": float(b["neg_score_mean"]),
        "finetuned_neg_score_mean": float(f["neg_score_mean"]),
    }
    for tfar in TARGET_FARS:
        col = f"tar_at_far_{tfar}"
        row[f"baseline_{col}"] = float(b[col])
        row[f"finetuned_{col}"] = float(f[col])
        row[f"{col}_delta"] = float(f[col] - b[col])
    comparison_rows.append(row)

comparison = pd.DataFrame(comparison_rows)
comparison_path = DRIVE_EVAL_SAVE_DIR / "baseline_vs_finetuned_medium_5epoch_lr5e5_comparison.csv"
comparison.to_csv(comparison_path, index=False)
display(comparison)
print("saved:", comparison_path)

meta = {
    "train_run_id": RUN_ID,
    "eval_run_id": EVAL_RUN_ID,
    "drive_data_dir": str(DRIVE_DATA_DIR),
    "data_zip_path": str(DATA_ZIP_PATH),
    "finetuned_nemo_path": str(FINETUNED_NEMO_PATH),
    "baseline_model_name": BASELINE_MODEL_NAME,
    "max_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_trials_per_enroll_sec": MAX_TRIALS_PER_ENROLL_SEC,
    "eval_trials_path": str(EVAL_TRIALS_PATH),
    "n_unique_audio_paths": len(unique_audio_paths),
    "baseline_embedding_failures": len(baseline_failures),
    "finetuned_embedding_failures": len(finetuned_failures),
    "train_save_dir": str(DRIVE_TRAIN_SAVE_DIR),
    "eval_save_dir": str(DRIVE_EVAL_SAVE_DIR),
}
meta_path = DRIVE_EVAL_SAVE_DIR / "eval_meta.json"
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(meta, ensure_ascii=False, indent=2))

,enroll_sec,baseline_eer,finetuned_medium_5epoch_lr5e5_eer,eer_delta_finetuned_minus_baseline,eer_improved,baseline_pos_score_mean,finetuned_pos_score_mean,baseline_neg_score_mean,finetuned_neg_score_mean,baseline_tar_at_far_0.01,finetuned_tar_at_far_0.01,tar_at_far_0.01_delta,baseline_tar_at_far_0.05,finetuned_tar_at_far_0.05,tar_at_far_0.05_delta,baseline_tar_at_far_0.1,finetuned_tar_at_far_0.1,tar_at_far_0.1_delta
0,1.5,0.220879,0.181300,-0.039578,True,0.396571,0.475262,0.214730,0.053444,0.239137,0.261912,0.022775,0.473479,0.534312,0.060833,0.613126,0.698232,0.085106
1,2.0,0.220979,0.181718,-0.039261,True,0.395308,0.475385,0.213193,0.055415,0.225150,0.242216,0.017066,0.460778,0.542814,0.082036,0.606587,0.703293,0.096707
2,2.5,0.219792,0.183402,-0.036390,True,0.403408,0.481379,0.216840,0.059759,0.245340,0.255262,0.009922,0.484666,0.552014,0.067348,0.616657,0.705352,0.088695
3,3.0,0.214654,0.176117,-0.038537,True,0.416314,0.495250,0.227214,0.058495,0.231440,0.263601,0.032161,0.491133,0.546138,0.055005,0.630598,0.709047,0.078449
4,4.0,0.203221,0.167209,-0.036013,True,0.438564,0.518902,0.238183,0.060374,0.286100,0.296908,0.010808,0.549385,0.591114,0.041729,0.673972,0.740919,0.066947
5,5.0,0.189939,0.158365,-0.031573,True,0.455135,0.532318,0.246949,0.063802,0.277628,0.315064,0.037436,0.527403,0.598682,0.071279,0.676550,0.753220,0.076670


saved: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_eval_runs/eval_medium_5epoch_lr5e5_20260502_070940/baseline_vs_finetuned_medium_5epoch_lr5e5_comparison.csv
{
  "train_run_id": "titanet_medium_5epoch_lr5e5_20260502_070940",
  "eval_run_id": "eval_medium_5epoch_lr5e5_20260502_070940",
  "drive_data_dir": "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터",
  "data_zip_path": "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_local_subset_medium.zip",
  "finetuned_nemo_path": "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_finetune_runs/titanet_medium_5epoch_lr5e5_20260502_070940/titanet_small/titanet_small_finetuned_final.nemo",
  "baseline_model_name": "titanet_small",
  "max_epochs": 5,
  "learning_rate": 5